In [3]:
import re
import pandas as pd

input_filename = "clean.csv"
output_filename = "business_data_cleaned.csv"

df_raw = pd.read_csv(input_filename)
initial_len = len(df_raw)
print(f"Размер датасета до очистки: {df_raw.shape}")

def clean_text(text):
    if not isinstance(text, str):
        return ""

    # Удаляем специфичный рекламный мусор с сайта
    text = re.sub(
        r"you have exhausted your monthly limit of free stories.*?sign in",
        "",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(
        r"register to read more stories.*?sign in", "", text, flags=re.IGNORECASE
    )
    text = re.sub(r"\badvertisement\b", "", text, flags=re.IGNORECASE)
    text = re.sub(r"pictwittercom\S+", "", text, flags=re.IGNORECASE)
    text = re.sub(r"http\S+", "", text)

    # Приведение к нижнему регистру
    text_lower = text.lower()

    # Удаление знаков препинания и спецсимволов (оставляем буквы, цифры и пробелы)
    cleaned = re.sub(r"[^a-z0-9\s]", "", text_lower)

    #  Схлопывание множественных пробелов в один
    single_spaced = re.sub(r"\s+", " ", cleaned)

    return single_spaced.strip()


def word_count(text):
    return len(str(text).split())

# 3. АВТООПРЕДЕЛЕНИЕ ЦЕЛЕВОЙ КОЛОНКИ С ТЕКСТОМ
if "text" in df_raw.columns:
    main_col = "text"
elif "content" in df_raw.columns:
    main_col = "content"
elif "description" in df_raw.columns:
    main_col = "description"
else:
    # Ищем текстовые колонки и выбираем ту, где в среднем больше всего символов
    text_cols = [col for col in df_raw.columns if df_raw[col].dtype == "object"]
    main_col = max(
        text_cols, key=lambda col: df_raw[col].astype(str).str.len().mean()
    )

print(f"Работаем со столбцом: '{main_col}'")

# Оставляем только нужную колонку и переименовываем её в 'text'
df = df_raw[[main_col]].rename(columns={main_col: "text"})

# 4. СТАДИИ ФИЛЬТРАЦИИ И ОЧИСТКИ

print("\nОчистка данных...")

# Удаляем пустые значения (NaN) на старте
df = df.dropna(subset=["text"])

# Применяем объединенную функцию очистки текста
df["text"] = df["text"].apply(clean_text)

# Избавляемся от пустых строк, которые могли образоваться после удаления мусора
df = df[df["text"] != ""]

# Удаляем повторяющиеся статьи (дубликаты)
df = df.drop_duplicates(subset=["text"])

# Отсекаем статьи короче 50 слов (мусор, недогруженные страницы, короткие анонсы)
df = df[df["text"].apply(word_count) > 50]

# Сбрасываем индексы строк по порядку
df = df.reset_index(drop=True)

print(f" Было: {initial_len} строк → Стало: {len(df)} строк после фильтрации")


# 5. СОХРАНЕНИЕ И ПРОВЕРКА РЕЗУЛЬТАТА
df.to_csv(output_filename, index=False)
print(f"Очищенный датасет успешно сохранен в файл '{output_filename}'!")

# Выводим превью первой строки для контроля качества
if len(df) > 0:
    print("\nПример финального очищенного текста из первой строки:")
    print(df["text"].iloc[0][:500] + "...")
else:
    print("\nВнимание: После фильтрации не осталось ни одной строки!")


FileNotFoundError: [Errno 2] No such file or directory: 'business_data.csv'

In [2]:
import os
print("Ваш файл лежит в папке:", os.getcwd())


Ваш файл лежит в папке: /home/anna/Univer/Proect/src
